In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.missed_calls_14_days AS (  
WITH deduped_id AS (
    SELECT
        lead_id,
        id AS activity_id,
        status,
        activity_at,
        date_updated,
        bronze_insert_date,
        ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed
    WHERE
        type = 'Call'
        AND direction = 'inbound'
        AND disposition = 'no-answer'
),
deduped_calls AS (
    SELECT *
    FROM deduped_id
    WHERE id_rn = 1
)
SELECT
    lead_id,
    COUNT(*) AS missed_calls_14_days
FROM
    deduped_calls
WHERE
    DATEDIFF(CURRENT_DATE, DATE(activity_at)) <= 14
GROUP BY
    lead_id
)